# Importamos librerias

In [1]:
!pip install -q supervision ultralytics trackers
import cv2
import supervision as sv
from ultralytics import YOLO, SAM
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics.models.sam import SAM3SemanticPredictor
#from ultralytics.models.sam import SAM3VideoSemanticPredictor
from trackers import ByteTrackTracker

#from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
from google.colab import userdata
import os
import json
import csv
from tqdm.auto import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# checamos que este activa la GPU

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

Usando dispositivo: cuda
GPU: Tesla T4


# usamos nuestro token de HugginFace para descargar SAM3

In [3]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

path = hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt"
)

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

# cargamos los pesos de SAM y los pesos de nuestro YOLO fine-tuneado

In [4]:
%%capture
path_sam = "/root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt"
sam_model = SAM(path_sam)
sam_model.to(DEVICE)

model = YOLO("/content/best.pt")
model.to(DEVICE)

#mask_annotator = sv.MaskAnnotator(opacity=0.6)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(
        text_scale=1.0,
        text_thickness=2,
        text_padding=5
)

# datos de la cancha y algunos annotators

In [5]:
SOURCE_POINTS = np.float32([
    [185, 235],
    [1075, 240],
    [1050, 1785],
    [40, 1550]
])

CAMPO_W, CAMPO_H = 364, 486
ESCALA_A_PX_CM = 2.0
TARGET_POINTS = np.float32([[0,0], [CAMPO_W, 0], [CAMPO_W, CAMPO_H], [0, CAMPO_H]])

H = cv2.getPerspectiveTransform(SOURCE_POINTS, TARGET_POINTS)

CLASS_NAMES = {0: "equipo_a", 1: "equipo_b", 2: "balon"}
COLORS_HEX  = {0: "#00b4d8", 1: "#ff4d4d", 2: "#ff9500"}
COLORS_BGR  = {0: (216, 180, 0), 1: (77, 77, 255), 2: (0, 149, 255)}

TEAM_PALETTE = sv.ColorPalette.from_hex([COLORS_HEX[0], COLORS_HEX[1], COLORS_HEX[2]])

ellipse_annotator = sv.EllipseAnnotator(
    color=TEAM_PALETTE,
    color_lookup=sv.ColorLookup.CLASS
)

triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex(COLORS_HEX[2]),
    color_lookup=sv.ColorLookup.CLASS,
    base=30,
    height=25
)

trace_annotator = sv.TraceAnnotator(
    color=sv.Color.from_hex(COLORS_HEX[2]),
    trace_length=30,
    thickness=2
)

SAM_IMGSZ = 1036

tracker = ByteTrackTracker(
    lost_track_buffer=90,
    frame_rate=30.0,
    track_activation_threshold=0.5,
    minimum_consecutive_frames=3,
    minimum_iou_threshold=0.05,
    high_conf_det_threshold=0.5,
)
tactical_log = []
MAX_ROBOTS = 4
last_ball_pos = None
track_streaks = {}

# variables para estado de marcador
GOAL_A_LINE = ((310, 1600), (700, 1700))
GOAL_B_LINE = ((410, 240), (890, 240))

score_a = 0
score_b = 0
last_ball_pixel_pos = None
last_goal_frame = -9999
GOAL_COOLDOWN_FRAMES = 60

# Funciones de apoyo para el pipeline

In [17]:
#---------------------------------------------------------------------------------------
def segment_boxes_sam(frame_bgr: np.ndarray, boxes: np.ndarray) -> sv.Detections:
    if boxes is None or len(boxes) == 0:
        return sv.Detections.empty()

    bboxes = boxes.tolist()
    sam_results = sam_model(frame_bgr, bboxes=bboxes, imgsz=SAM_IMGSZ,device=DEVICE, half=(DEVICE=="cuda"), verbose=False)[0]
    sam_detections = sv.Detections.from_ultralytics(sam_results)
    return sam_detections
#------------------------------------------------------------------------------------

def compute_iou_matrix(boxes_a, boxes_b):
    boxes_a = np.array(boxes_a, dtype=np.float32)
    boxes_b = np.array(boxes_b, dtype=np.float32)

    ax1, ay1, ax2, ay2 = boxes_a[:, 0], boxes_a[:, 1], boxes_a[:, 2], boxes_a[:, 3]
    bx1, by1, bx2, by2 = boxes_b[:, 0], boxes_b[:, 1], boxes_b[:, 2], boxes_b[:, 3]

    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)

    inter_x1 = np.maximum(ax1[:, None], bx1[None, :])
    inter_y1 = np.maximum(ay1[:, None], by1[None, :])
    inter_x2 = np.minimum(ax2[:, None], bx2[None, :])
    inter_y2 = np.minimum(ay2[:, None], by2[None, :])

    inter_w = np.clip(inter_x2 - inter_x1, 0, None)
    inter_h = np.clip(inter_y2 - inter_y1, 0, None)
    inter_area = inter_w * inter_h

    union = area_a[:, None] + area_b[None, :] - inter_area
    return np.where(union > 0, inter_area / union, 0)

#------------------------------------------------------------------------------------
def assign_class_by_iou(dets_sam, robot_candidates, iou_thresh=0.3):
    if len(dets_sam) == 0 or len(robot_candidates) == 0:
        return sv.Detections.empty()

    iou_matrix = compute_iou_matrix(dets_sam.xyxy, robot_candidates.xyxy)
    best_match = iou_matrix.argmax(axis=1)
    best_iou = iou_matrix.max(axis=1)

    class_id = np.full(len(dets_sam), -1, dtype=int)
    confidence = np.zeros(len(dets_sam), dtype=np.float32)

    for i in range(len(dets_sam)):
        if best_iou[i] >= iou_thresh:
            class_id[i] = robot_candidates.class_id[best_match[i]]
            confidence[i] = robot_candidates.confidence[best_match[i]]

    keep = class_id != -1
    if not keep.any():
        return sv.Detections.empty()

    dets_sam = dets_sam[keep]
    dets_sam.class_id = class_id[keep]
    dets_sam.confidence = confidence[keep]
    return dets_sam

#------------------------------------------------------------------------------------

def update_track_streaks(dets_sam: sv.Detections):
    seen = set()
    if dets_sam.tracker_id is not None:
        for tid in dets_sam.tracker_id:
            tid = int(tid)
            seen.add(tid)
            track_streaks[tid] = track_streaks.get(tid, 0) + 1

    for tid in list(track_streaks.keys()):
        if tid not in seen:
            track_streaks[tid] = 0

#------------------------------------------------------------------------------------

def limit_robot_count(dets_sam: sv.Detections, max_robots: int = MAX_ROBOTS) -> sv.Detections:
    if dets_sam.class_id is None:
        return dets_sam

    robot_idx = np.where(dets_sam.class_id != 2)[0]
    other_idx = np.where(dets_sam.class_id == 2)[0]

    if len(robot_idx) <= max_robots:
        return dets_sam

    scores = []
    for i in robot_idx:
        tid = int(dets_sam.tracker_id[i]) if dets_sam.tracker_id is not None else None
        streak = track_streaks.get(tid, 0) if tid is not None else 0
        area = np.count_nonzero(dets_sam.mask[i]) if dets_sam.mask is not None else 0
        scores.append((streak, area))

    order = sorted(range(len(robot_idx)), key=lambda k: scores[k], reverse=True)
    keep_robot_idx = robot_idx[order[:max_robots]]

    keep = np.sort(np.concatenate([keep_robot_idx, other_idx]))
    return dets_sam[keep]
#------------------------------------------------------------------------------------

def mask_centroid_projected(mask, H):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    cx = xs.mean()
    cy = ys.mean()
    pt = np.float32([[[cx, cy]]])
    proj = cv2.perspectiveTransform(pt, H)
    return float(proj[0][0][0]), float(proj[0][0][1])

#---------------------------------------------------------------------------------------
def limit_ball_count(dets_sam: sv.Detections, H: np.ndarray) -> sv.Detections:
    global last_ball_pos

    if dets_sam.class_id is None:
        return dets_sam

    ball_idx = np.where(dets_sam.class_id == 2)[0]
    other_idx = np.where(dets_sam.class_id != 2)[0]

    if len(ball_idx) == 0:
        return dets_sam

    if len(ball_idx) == 1:
        pos = mask_centroid_projected(dets_sam.mask[ball_idx[0]], H)
        if pos is not None:
            last_ball_pos = pos
        return dets_sam

    positions = [mask_centroid_projected(dets_sam.mask[i], H) for i in ball_idx]

    if last_ball_pos is not None:
        dists = [
            np.hypot(p[0] - last_ball_pos[0], p[1] - last_ball_pos[1]) if p is not None else np.inf
            for p in positions
        ]
        best = int(np.argmin(dists))
    else:
        areas = [np.count_nonzero(dets_sam.mask[i]) for i in ball_idx]
        best = int(np.argmax(areas))

    keep_ball_idx = ball_idx[best]
    if positions[best] is not None:
        last_ball_pos = positions[best]

    keep = np.sort(np.concatenate([other_idx, [keep_ball_idx]]))
    return dets_sam[keep]

#------------------------------------------------------------------------------------

def segments_intersect(p1, p2, p3, p4):
    def ccw(a, b, c):
        return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
    return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)

#------------------------------------------------------------------------------------

def check_goal(dets_sam: sv.Detections, frame_idx: int) -> str | None:
    global score_a, score_b, last_ball_pixel_pos, last_goal_frame

    if dets_sam.class_id is None:
        return None

    ball_idx = np.where(dets_sam.class_id == 2)[0]
    if len(ball_idx) == 0:
        return None

    mask = dets_sam.mask[ball_idx[0]] if dets_sam.mask is not None else None
    if mask is None or not mask.any():
        return None

    ys, xs = np.where(mask)
    current_pos = (float(xs.mean()), float(ys.mean()))

    goal_scored = None

    if last_ball_pixel_pos is not None and (frame_idx - last_goal_frame) > GOAL_COOLDOWN_FRAMES:
        dy = current_pos[1] - last_ball_pixel_pos[1]

        if dy > 0 and segments_intersect(last_ball_pixel_pos, current_pos, *GOAL_A_LINE):
            score_b += 1
            last_goal_frame = frame_idx
            goal_scored = "B"
        elif dy < 0 and segments_intersect(last_ball_pixel_pos, current_pos, *GOAL_B_LINE):
            score_a += 1
            last_goal_frame = frame_idx
            goal_scored = "A"

    last_ball_pixel_pos = current_pos
    return goal_scored

#------------------------------------------------------------------------------------

def draw_goal_lines(frame_bgr: np.ndarray) -> np.ndarray:
    cv2.line(frame_bgr, GOAL_A_LINE[0], GOAL_A_LINE[1], COLORS_BGR[0], 4)
    cv2.line(frame_bgr, GOAL_B_LINE[0], GOAL_B_LINE[1], COLORS_BGR[1], 4)
    return frame_bgr

#------------------------------------------------------------------------------------

def draw_scoreboard(frame_bgr: np.ndarray) -> np.ndarray:
    h, w = frame_bgr.shape[:2]
    text = f"A {score_a}  -  {score_b} B"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
    x = (w - tw) // 2
    y = th + 30
    cv2.rectangle(frame_bgr, (x - 20, y - th - 20), (x + tw + 20, y + 15), (0, 0, 0), -1)
    cv2.putText(frame_bgr, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
    return frame_bgr

#------------------------------------------------------------------------------------
def detections_to_dict(detections, frame_idx=None, class_names=None):
    return {
        "frame_idx": frame_idx,
        "xyxy": detections.xyxy.tolist(),
        "confidence": detections.confidence.tolist() if detections.confidence is not None else None,
        "class_id": detections.class_id.tolist() if detections.class_id is not None else None,
        "class_names": [
            class_names.get(int(c), str(c))
            for c in detections.class_id
        ] if class_names and detections.class_id is not None else None,
        "has_mask": detections.mask is not None,
    }

#------------------------------------------------------------------------------------
def detections_to_tactical_records(dets_sam: sv.Detections, H: np.ndarray, frame_idx: int,
                                  class_names: dict | None = None, team_map: dict | None = None) -> list[dict]:
    records = []
    n = len(dets_sam)

    class_ids = dets_sam.class_id if dets_sam.class_id is not None else [None] * n
    confidences = dets_sam.confidence if dets_sam.confidence is not None else [None] * n
    tracker_ids = dets_sam.tracker_id if dets_sam.tracker_id is not None else [None] * n
    masks = dets_sam.mask if dets_sam.mask is not None else [None] * n

    for i in range(n):
        cid = class_ids[i]
        conf = confidences[i]
        tid = tracker_ids[i]
        mask = masks[i]

        class_id = int(cid) if cid is not None else None
        confidence = float(conf) if conf is not None else None
        track_id = int(tid) if tid is not None else None

        class_name = (class_names.get(class_id, str(class_id)) if class_names else str(class_id)) if class_id is not None else None
        team = team_map.get(track_id) if team_map else None

        if mask is None or not mask.any():
            x = None
            y = None
            mask_area = 0
        else:
            ys, xs = np.where(mask)
            cx = xs.mean()
            cy = ys.mean()

            pt = np.float32([[[cx, cy]]])
            proj = cv2.perspectiveTransform(pt, H)

            x = float(proj[0][0][0])
            y = float(proj[0][0][1])
            mask_area = int(mask.sum())

        records.append({
            "frame_idx": frame_idx,
            "track_id": track_id,
            "class_id": class_id,
            "class_name": class_name,
            "team": team,
            "confidence": confidence,
            "x": x,
            "y": y,
            "mask_area": mask_area
        })

    return records

# Mapa tactico

In [7]:
def project_mask_contour(mask: np.ndarray, H: np.ndarray) -> np.ndarray | None:
    """Poryecta el contorno exterior de una mascara al campo canonico"""
    contours, _ = cv2.findContours(mask.astype(np.uint8),
                                  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 1, 2).astype(np.float32)
    pts_proj = cv2.perspectiveTransform(pts, H)
    return pts_proj.reshape(-1, 2).astype(np.int32)

def draw_tactical_with_masks(dets_sam: sv.Detections, H: np.ndarray, campo_w: int = CAMPO_W, campo_h: int = CAMPO_H, vertical: bool = True) -> np.ndarray:
    """Dibuja el mapa tactico con rellenos y contornos de mascaras SAM proyectadas"""
    canvas = np.zeros((campo_h, campo_w, 3), dtype=np.uint8)
    canvas[:] = (50, 67, 27)

    # Líneas del campo
    cv2.rectangle(canvas, (0, 0), (campo_w - 1, campo_h - 1), (120, 200, 116), 2)
    cv2.line(canvas, (0, campo_h // 2), (campo_w, campo_h // 2), (120, 200, 116), 1)
    cv2.circle(canvas, (campo_w // 2, campo_h // 2), int(30 * ESCALA_A_PX_CM), (120, 200, 116), 1)
    pen_w = int(80 * ESCALA_A_PX_CM); pen_h = int(40 * ESCALA_A_PX_CM)
    pen_x = (campo_w - pen_w) // 2
    cv2.rectangle(canvas, (pen_x, 0), (pen_x + pen_w, pen_h), (120, 200, 116), 1)
    cv2.rectangle(canvas, (pen_x, campo_h - pen_h),
                           (pen_x + pen_w, campo_h - 1), (120, 200, 116), 1)
    goal_wp = int(60 * ESCALA_A_PX_CM); goal_xp = (campo_w - goal_wp) // 2
    cv2.rectangle(canvas, (goal_xp, 0), (goal_xp + goal_wp, int(8 * ESCALA_A_PX_CM)), (0, 214, 255), 2)
    cv2.rectangle(canvas, (goal_xp, campo_h - int(8 * ESCALA_A_PX_CM)),
                           (goal_xp + goal_wp, campo_h - 1), (216, 180, 0), 2)

    masks     = dets_sam.mask
    class_ids = (dets_sam.class_id if dets_sam.class_id is not None
                 else np.zeros(len(dets_sam), dtype=int))

    if masks is not None:
        for m, cid in zip(masks, class_ids):
            color = COLORS_BGR.get(int(cid), (200, 200, 200))

            # opcion A: relleno semitransparente (warp de la mascara)
            m_uint8 = m.astype(np.uint8) * 255
            m_warped = cv2.warpPerspective(m_uint8, H, (campo_w, campo_h))
            overlay = canvas.copy()
            overlay[m_warped > 0] = color
            canvas = cv2.addWeighted(overlay, 0.45, canvas, 0.55, 0)

            # opcion B: contorno proyectado
            contorno = project_mask_contour(m, H)
            if contorno is not None and len(contorno) > 2:
                cv2.polylines(canvas, [contorno], True, color, 2)

            # Centroide de la máscara
            ys, xs = np.where(m)
            if len(xs) > 0:
                cx = int(xs.mean()); cy = int(ys.mean())
                pt = np.float32([[[cx, cy]]])
                proj = cv2.perspectiveTransform(pt, H)
                px, py = int(proj[0][0][0]), int(proj[0][0][1])
                if 0 <= px < campo_w and 0 <= py < campo_h:
                    cv2.circle(canvas, (px, py), 8, color, -1)
                    cv2.circle(canvas, (px, py), 8, (255, 255, 255), 1)

    if not vertical:
        canvas = cv2.rotate(canvas, cv2.ROTATE_90_CLOCKWISE)

    return canvas

# funcion para sobreponer el mapa tactico sobre el video

In [8]:
def overlay_tactical_map(frame_bgr: np.ndarray, tactical_bgr: np.ndarray, scale: float = 0.35, position: str = "bottom_center", alpha: float = 0.9, margin: int = 20) -> np.ndarray:
  output = frame_bgr.copy()
  frame_h, frame_w = output.shape[:2]
  tact_h, tact_w = tactical_bgr.shape[:2]

  # redimensionamos manteniendo proporcion
  new_w = int(frame_w * scale)
  new_h = int(tact_h * (new_w / tact_w))

  tactical_resized = cv2.resize(
      tactical_bgr,
      (new_w, new_h),
      interpolation=cv2.INTER_AREA
  )

  # calculamos posicion
  if position == "center":
    x = (frame_w - new_w) // 2
    y = (frame_h, new_h) // 2

  elif position == "bottom_center":
    x = (frame_w - new_w) // 2
    y = frame_h - new_h - margin

  elif position == "top_left":
    x = margin
    y = margin

  else:
    raise ValueError("position debe ser 'center', 'bottom_center' o 'top_left'. ")

  # evitar salir del frame
  x = max(0, min(x, frame_w - new_w))
  y = max(0, min(y, frame_h - new_h))

  roi = output[y:y + new_h, x:x + new_w]
  blended = cv2.addWeighted(tactical_resized, alpha, roi, 1 - alpha, 0)
  output[y:y + new_h, x:x + new_w] = blended
  return output

# procesamiento a cada frame en el video

In [9]:
def procesar_frame_base(frame: np.ndarray, frame_idx: int, vertical: bool = True):

    results = model(frame, conf=0.4, iou=0.5, verbose=False)[0]
    dets = sv.Detections.from_ultralytics(results)

    warped = cv2.warpPerspective(frame, H, (CAMPO_W, CAMPO_H))

    if len(dets) == 0:
        empty_dets = sv.Detections.empty()
        tactical = draw_tactical_with_masks(
            empty_dets,
            H,
            vertical=vertical
        )
        return {
            "dets_sam": sv.Detections.empty(),
            "warped": warped,
            "tactical": tactical,
        }

    dets_sam = segment_boxes_sam(frame, dets.xyxy)

    if len(dets_sam) > 0:
      dets_sam = assign_class_by_iou(dets_sam, dets)

    if len(dets_sam) == 0:
      tactical = draw_tactical_with_masks(sv.Detections.empty(), H, vertical=vertical)
      return{
          "dets_sam": sv.Detections.empty(),
          "warped": warped,
          "tactical": tactical,
      }

    dets_sam = tracker.update(dets_sam)
    update_track_streaks(dets_sam)
    dets_sam = limit_robot_count(dets_sam)
    dets_sam = limit_ball_count(dets_sam, H)
    goal_scored = check_goal(dets_sam, frame_idx)

    records = detections_to_tactical_records(dets_sam, H, frame_idx, class_names=CLASS_NAMES)
    tactical_log.extend(records)

    tactical = draw_tactical_with_masks(dets_sam, H, vertical=vertical)
    result = {
        "dets_sam": dets_sam,
        "warped": warped,
        "tactical": tactical,
        "goal_scored": goal_scored,
    }

    return result

# funcion para aplicar las mascaras en cada frame del video

In [10]:
def propagacion_frames_overlay(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    data = procesar_frame_base(
        frame,
        frame_idx,
        vertical=False
    )

    dets_sam = data["dets_sam"]
    tactical = data["tactical"]

    annotated = frame.copy()
    annotated = draw_goal_lines(annotated)

    if dets_sam.class_id is not None:
        robot_dets = dets_sam[dets_sam.class_id != 2]
        ball_dets = dets_sam[dets_sam.class_id == 2]
    else:
        robot_dets = sv.Detections.empty()
        ball_dets = sv.Detections.empty()

    if len(robot_dets) > 0:
        annotated = ellipse_annotator.annotate(
            scene=annotated,
            detections=robot_dets
        )
        robot_labels = [CLASS_NAMES.get(int(cid), str(cid)) for cid in robot_dets.class_id]
        annotated = label_annotator.annotate(
            scene=annotated,
            detections=robot_dets,
            labels=robot_labels
        )

    if len(ball_dets) > 0:
        annotated = trace_annotator.annotate(
            scene=annotated,
            detections=ball_dets
        )
        annotated = triangle_annotator.annotate(
            scene=annotated,
            detections=ball_dets
        )


    annotated = draw_scoreboard(annotated)

    frame_overlay = overlay_tactical_map(
        frame_bgr=annotated,
        tactical_bgr=tactical,
        scale=0.35,
        position="bottom_center"
    )

    return frame_overlay

# funcion para regresar el video original junto con el mapa cenital y el resultado segmentado.

In [11]:
def propagacion_frames_overlay_full(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    data = procesar_frame_base(
        frame,
        frame_idx,
        vertical=False
    )

    dets_sam = data["dets_sam"]
    warped = data["warped"]
    tactical = data["tactical"]

    annotated = frame.copy()
    annotated = draw_goal_lines(annotated)

    if dets_sam.class_id is not None:
        robot_dets = dets_sam[dets_sam.class_id != 2]
        ball_dets = dets_sam[dets_sam.class_id == 2]
    else:
        robot_dets = sv.Detections.empty()
        ball_dets = sv.Detections.empty()

    if len(robot_dets) > 0:
        annotated = ellipse_annotator.annotate(
            scene=annotated,
            detections=robot_dets
        )
        robot_labels = [CLASS_NAMES.get(int(cid), str(cid)) for cid in robot_dets.class_id]
        annotated = label_annotator.annotate(
            scene=annotated,
            detections=robot_dets,
            labels=robot_labels
        )

    if len(ball_dets) > 0:
        annotated = trace_annotator.annotate(
            scene=annotated,
            detections=ball_dets
        )
        annotated = triangle_annotator.annotate(
            scene=annotated,
            detections=ball_dets
        )

    annotated = draw_scoreboard(annotated)

    frame_overlay = overlay_tactical_map(
        frame_bgr=annotated,
        tactical_bgr=tactical,
        scale=0.35,
        position="bottom_center"
    )

    target_h = frame.shape[0]

    warped_h, warped_w = warped.shape[:2]
    warped_scaled_w = int(warped_w * (target_h / warped_h))
    warped_resized = cv2.resize(warped, (warped_scaled_w, target_h))

    return np.hstack([frame, warped_resized, frame_overlay])

# llamar a la funcion de propagacion para cada frame y descarga de video

In [12]:
def procesar_video(source_video: str, output_video: str, callback):
    cap = cv2.VideoCapture(source_video)

    if not cap.isOpened():
        raise ValueError(f"No se pudo abrir: {source_video}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ret, frame = cap.read()
    if not ret:
        raise ValueError("No se pudo leer el primer frame")

    first_output = callback(frame, 0)
    out_h, out_w = first_output.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        output_video,
        fourcc,
        fps,
        (out_w, out_h)
    )

    if not writer.isOpened():
        raise ValueError("No se pudo abrir el VideoWriter")

    writer.write(first_output)

    frame_idx = 1

    with tqdm(total=total_frames, desc=f'Porcesando {output_video}', unit='frame') as pbar:
        pbar.update(1)

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            output = callback(frame, frame_idx)

            if output.shape[:2] != (out_h, out_w):
                output = cv2.resize(output, (out_w, out_h))

            writer.write(output)

            frame_idx += 1
            pbar.update(1)

    cap.release()
    writer.release()

    print(f"Video guardado: {output_video}")

# inferencia sobre el video

In [13]:
SOURCE_VIDEO = "/content/video_prueba_final.mp4"

In [18]:
tactical_log.clear()
score_a = 0
score_b = 0
last_ball_pixel_pos = None
last_goal_frame = -9999

procesar_video(
    SOURCE_VIDEO,
    "futbot_overlay_2.mp4",
   propagacion_frames_overlay_full
)

Porcesando futbot_overlay_2.mp4:   0%|          | 0/3640 [00:00<?, ?frame/s]

Video guardado: futbot_overlay_2.mp4


# guardamos datos de las posiciones de los robots para analisis

In [19]:
import csv

with open("tactical_log.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=tactical_log[0].keys())
    writer.writeheader()
    writer.writerows(tactical_log)